## Semantic Chunking

In [1]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

In [3]:
model=SentenceTransformer('all-MiniLM-L6-v2')
## Sample text
text="""
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

sentences=[s.strip() for s in text.split("\n") if s.strip()]

embeddings=model.encode(sentences)

threshold=0.7
chunks=[]
current_chunk=[sentences[0]]

for i in range(1, len(sentences)):
    sim = cosine_similarity(
        [embeddings[i - 1]],
        [embeddings[i]]
    )[0][0]

    if sim>=threshold:
        current_chunk.append(sentences[i])
    else:
        chunks.append(" ".join(current_chunk))
        current_chunk=[sentences[i]]

chunks.append(" ".join(current_chunk))

print("\n Semantic Chunks:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk}")


 Semantic Chunks:
Chunk 1: LangChain is a framework for building applications with LLMs. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
Chunk 2: You can create chains, agents, memory, and retrievers.
Chunk 3: The Eiffel Tower is located in Paris.
Chunk 4: France is a popular tourist destination.


Rag Pipeline Modular Coding

In [6]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from langchain_classic.schema import Document
from langchain_classic.vectorstores import FAISS
# from langchain_openai import OpenAIEmbeddings
from langchain_ollama import OllamaEmbeddings,ChatOllama
from langchain.chat_models import init_chat_model
from langchain_classic.schema.runnable import RunnableLambda, RunnableMap
from langchain_classic.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_groq import ChatGroq
import os
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")


In [7]:
class ThresholdSemanticChunker:
    def __init__(self, model_name="all-MiniLM-L6-v2", threshold=0.7):
        self.model = SentenceTransformer(model_name)
        self.threshold = threshold

    def split(self, text:str):
        sentences = [s.strip() for s in text.split("\n") if s.strip()]
        embeddings = self.model.encode(sentences)
        chunks = []
        current_chunk = [sentences[0]]
        for i in range(1, len(sentences)):
            sim = cosine_similarity([embeddings[i - 1]], [embeddings[i]])[0][0]
            if sim >= self.threshold:
                current_chunk.append(sentences[i])
            else:
                chunks.append(". ".join(current_chunk) + ".")
                current_chunk = [sentences[i]]

        chunks.append(". ".join(current_chunk) + ".")
        return chunks
    def split_documents(self, documents:list[Document]):
        result = []
        for doc in documents:
            for chunk in self.split(doc.page_content):
                result.append(Document(page_content=chunk, metadata=doc.metadata))
        return result

In [8]:
# Sample text
sample_text = """
LangChain is a framework for building applications with LLMs.
Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.
You can create chains, agents, memory, and retrievers.
The Eiffel Tower is located in Paris.
France is a popular tourist destination.
"""

doc = Document(page_content=sample_text)
doc

Document(metadata={}, page_content='\nLangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.\n')

In [9]:
chunker = ThresholdSemanticChunker(threshold=0.7)
chunks = chunker.split_documents([doc])
print("\nSemantic Chunks:")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk.page_content}")


Semantic Chunks:
Chunk 1: LangChain is a framework for building applications with LLMs.. Langchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone..
Chunk 2: You can create chains, agents, memory, and retrievers..
Chunk 3: The Eiffel Tower is located in Paris..
Chunk 4: France is a popular tourist destination..


In [10]:
embeddings=OllamaEmbeddings(model="llama3.2:latest")
vectorstore=FAISS.from_documents(chunks, embeddings)
retriever=vectorstore.as_retriever()

In [11]:
## Prompt Template

# --- 5. Prompt Template ---
template = """Answer the question based on the following context:

{context}

Question: {question}
"""

prompt = PromptTemplate.from_template(template)
prompt

PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Answer the question based on the following context:\n\n{context}\n\nQuestion: {question}\n')

In [21]:
llm =ChatOllama(model="llama3.2:latest")
rag_chain = (RunnableMap(
    {
        "context":lambda x: retriever.invoke(x["question"]) ,
        "question": lambda x: x,
    }
)|prompt|llm|StrOutputParser()
)


In [22]:

# --- 8. Run Query ---
query = {"question": "What is LangChain used for?"}
result = rag_chain.invoke(query)

print(result)

Based on the provided context, it appears that LangChain is a framework used for building applications with Large Language Models (LLMs). The exact details of its capabilities are not specified in this snippet, but it mentions providing modular abstractions to combine LLMs with other tools like OpenAI and Pinecone.


## Sematic Chunker with Langchain

In [28]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_classic.document_loaders import TextLoader


In [ ]:
loader=TextLoader("./langchain_intro.txt")
# chunks=loader.load()
docs=loader.load()
# ! embedding
embeddings=OllamaEmbeddings(model="llama3.2:latest")
chunker=SemanticChunker(embeddings=embeddings)
chunks=chunker.split_documents(docs)

for i , chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {chunk.page_content}\n")
    

[Document(metadata={'source': './langchain_intro.txt'}, page_content='LangChain is a framework for building applications with LLMs.\nLangchain provides modular abstractions to combine LLMs with tools like OpenAI and Pinecone.\nYou can create chains, agents, memory, and retrievers.\nThe Eiffel Tower is located in Paris.\nFrance is a popular tourist destination.')]